In [ ]:
# --- ADDED NEW FEATURE: Local execution fallback ---
import sys
from unittest.mock import MagicMock
if 'google.colab' not in sys.modules:
    sys.modules['google'] = MagicMock()
    sys.modules['google.colab'] = MagicMock()
    sys.modules['google.colab.drive'] = MagicMock()
    
    # Mock os.makedirs to redirect /content/... to ./content/...
    import builtins, os
    _orig_makedirs = os.makedirs
    def _mock_makedirs(name, mode=511, exist_ok=False):
        if str(name).startswith('/content/'):
            name = str(name).replace('/content/', './content/')
        return _orig_makedirs(name, mode, exist_ok)
    os.makedirs = _mock_makedirs
    
    _orig_open = builtins.open
    def _mock_open(file, *args, **kwargs):
        if str(file).startswith('/content/'):
            file = str(file).replace('/content/', './content/')
        return _orig_open(file, *args, **kwargs)
    builtins.open = _mock_open
    
    # Also patch torch.save to handle paths
    import torch
    _orig_save = torch.save
    def _mock_save(obj, f, *args, **kwargs):
        if isinstance(f, str) and f.startswith('/content/'):
            f = f.replace('/content/', './content/')
        return _orig_save(obj, f, *args, **kwargs)
    torch.save = _mock_save


# 🔋 EVolvAI — Physics-Informed Generative Counterfactual VAE
## Academic SOTA Edition · Google Colab (CPU or GPU)

This notebook contains the complete, reproducible training pipeline for the EVolvAI research paper. 

### Methodological Highlights:
1. **Self-Attention TCN:** Fuses temporal causal convolutions with Multi-head Self-Attention to capture complex global grid dependencies.
2. **Data Augmentation Engine:** Uses Spatial Cloning and Temporal Shift-Scaling to synthetically expand 25 physical chargers to a 32-node grid topology.
3. **Resilient Drive Checkpointing:** Automatically mounts Google Drive, saves optimizer/model states periodically, and auto-resumes if the Colab runtime disconnects.

> 💡 **Usage:** Run all cells with `Ctrl+F9`. If the runtime disconnects, simply reconnect and run all again; training will resume from the last saved epoch.

In [ ]:
CFG.EPOCHS = 2
CFG.NUM_SYNTHETIC_DAYS = 50
print('Overriding EPOCHS to 2 to train quickly')

## 1 · Mount Google Drive & Setup Environment

In [ ]:
import sys
import os
from google.colab import drive

# Mount Drive for persistent checkpointing
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/EVolvAI_Research'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"\n💾 Checkpoints will be saved to: {DRIVE_DIR}")

!pip install -q torch numpy pandas pyarrow scipy requests

import json
import torch
import requests
import datetime
import pandas as pd
import numpy as np
import urllib.parse
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

print(f"🐍 Python : {sys.version.split()[0]}")
print(f"🔥 PyTorch: {torch.__version__}")

## 2 · Fetch Real-World Data (ACN + Open-Meteo)

In [ ]:
# --- A. ACN-DATA API EXTRACTION ---
API_TOKEN = '-RQtRLh5YnSZMVsanUCuLAa1DGvtdX35Q2qt5Dv5FA8'
HEADERS = {'Authorization': f'Bearer {API_TOKEN}'}
SITE = 'caltech'

print("📥 Fetching real charging data from ACN API (Jan-Feb 2019)...")
query = {
    "connectionTime": {
        "$gte": "Tue, 01 Jan 2019 00:00:00 GMT",
        "$lte": "Thu, 28 Feb 2019 23:59:59 GMT"
    }
}
encoded_where = urllib.parse.quote(json.dumps(query))
url_acn = f"https://ev.caltech.edu/api/v1/sessions/{SITE}?where={encoded_where}"

try:
    res_acn = requests.get(url_acn, headers=HEADERS)
    res_acn.raise_for_status()
    sessions = res_acn.json().get('_items', [])
    print(f"   ✓ Fetched {len(sessions)} charging sessions.")
    
    records = []
    for s in sessions:
        if not s.get('connectionTime') or not s.get('kWhDelivered'): continue
        conn_time = pd.to_datetime(s['connectionTime'])
        records.append({
            'date': conn_time.date().isoformat(),
            'hour': conn_time.hour,
            'node_id': hash(s.get('spaceID', 'unknown')) % 32, 
            'demand_kw': s['kWhDelivered'] / max((s.get('duration', 1) / 60), 1)
        })
    
    df_ev = pd.DataFrame(records, columns=['date', 'hour', 'node_id', 'demand_kw'])
    os.makedirs('data/processed', exist_ok=True)
    df_ev.to_parquet('data/processed/train_data.parquet')
except Exception as e:
    print(f"   ❌ API Fetch Failed: {e}")

# --- B. OPEN-METEO WEATHER EXTRACTION ---
print("\n📥 Fetching historical weather for Pasadena, CA...")
url_meteo = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 34.1478, "longitude": -118.1445, 
    "start_date": "2019-01-01", "end_date": "2019-02-28",
    "hourly": ["temperature_2m", "precipitation", "shortwave_radiation"],
    "timezone": "America/Los_Angeles"
}

try:
    res_meteo = requests.get(url_meteo, params=params)
    res_meteo.raise_for_status()
    w_data = res_meteo.json()

    df_w = pd.DataFrame({
        "datetime": pd.to_datetime(w_data["hourly"]["time"]),
        "temperature_c": w_data["hourly"]["temperature_2m"],
        "precipitation_mm": w_data["hourly"]["precipitation"],
        "solar_radiation": w_data["hourly"]["shortwave_radiation"]
    })
    df_w['date'] = df_w['datetime'].dt.date.astype(str)
    df_w['hour'] = df_w['datetime'].dt.hour
    df_w["solar_availability"] = df_w["solar_radiation"] / df_w["solar_radiation"].max()
    df_w["traffic_index"] = 0.5  # Placeholder for OSMnx data
    
    df_w = df_w[['date', 'hour', 'temperature_c', 'precipitation_mm', 'solar_availability', 'traffic_index']]
    df_w.to_parquet('data/processed/weather_data.parquet')
    print(f"   ✓ Fetched {len(df_w)} hours of climate data.")
except Exception as e:
    print(f"   ❌ Weather API Failed: {e}")

## 3 · Hyperparameters & Configuration

In [ ]:
class CFG:
    NUM_NODES = 32
    SEQ_LEN = 24
    NUM_WEATHER_FEATURES = 4
    NUM_FEATURES = NUM_NODES + NUM_WEATHER_FEATURES
    
    # Stage 1 Training: Physics Penalties Zeroed out
    LAMBDA_VOLT = 0.0
    LAMBDA_THERMAL = 0.0
    LAMBDA_XFMR = 0.0
    
    # Training Params
    EPOCHS = 150             
    BATCH_SIZE = 64
    LEARNING_RATE = 1e-4
    GRAD_CLIP_NORM = 1.0
    
    # Academic SOTA Model Capacity
    TCN_CHANNELS = [256, 512, 512, 512, 512, 512]
    KERNEL_SIZE = 3
    DROPOUT = 0.20           
    LATENT_DIM = 256
    COND_DIM = 6
    DECODER_HIDDEN = 1024
    
    CHECKPOINT_FILE = f"{DRIVE_DIR}/gcvae_sota_checkpoint.pt"

## 4 · Data Augmentation Engine & Loader

In [ ]:
class EVDemandDataset(Dataset):
    def __init__(self):
        print("\n⚙️ Fusing ACN Demand Data with Open-Meteo Climate Data...")
        
        use_fallback = True
        if os.path.exists("data/processed/train_data.parquet") and os.path.exists("data/processed/weather_data.parquet"):
            df_ev = pd.read_parquet("data/processed/train_data.parquet")
            df_w = pd.read_parquet("data/processed/weather_data.parquet")
            
            if not df_ev.empty and not df_w.empty:
                pivot_ev = df_ev.pivot_table(index=["date", "hour"], columns="node_id", values="demand_kw", fill_value=0.0)
                
                days, conditions, weather_features = [], [], []
                monthly_mean_temp = df_w['temperature_c'].mean()
                
                for date in pivot_ev.index.get_level_values('date').unique():
                    day_demand = pivot_ev.loc[date].values
                    day_weather = df_w[df_w['date'] == date].sort_values('hour')
                    
                    if len(day_demand) == CFG.SEQ_LEN and len(day_weather) == CFG.SEQ_LEN:
                        # A. Spatial Augmentation (Pad to 32 Nodes)
                        if day_demand.shape[1] < CFG.NUM_NODES:
                            needed = CFG.NUM_NODES - day_demand.shape[1]
                            clones = day_demand[:, np.random.choice(day_demand.shape[1], needed, replace=True)]
                            noise = np.random.normal(0, 0.05 * clones.std() + 1e-4, clones.shape)
                            day_demand = np.concatenate([day_demand, np.clip(clones + noise, 0, None)], axis=-1)
                        
                        days.append(day_demand)
                        w_tensor = day_weather[['temperature_c', 'precipitation_mm', 'solar_availability', 'traffic_index']].values
                        weather_features.append(w_tensor)
                        
                        # B. Dynamic Causal Condition Vector (C)
                        dt_obj = pd.to_datetime(date)
                        temp_anomaly = (day_weather['temperature_c'].mean() - monthly_mean_temp) / (day_weather['temperature_c'].std() + 1e-5)
                        conditions.append([
                            float(temp_anomaly), 1.0, float(day_weather['solar_availability'].mean()),
                            1.0 if dt_obj.weekday() >= 5 else 0.0, 0.0, float(day_weather['traffic_index'].mean())
                        ])
                
                raw_demand = np.stack(days)
                raw_conditions = np.stack(conditions).astype(np.float32)
                raw_weather = np.stack(weather_features)
                use_fallback = False
                
        if use_fallback:
            print("⚠️ Fallback: Using synthetic arrays.")
            raw_demand = np.random.uniform(0, 50, (100, CFG.SEQ_LEN, CFG.NUM_NODES))
            raw_conditions = np.zeros((100, CFG.COND_DIM), dtype=np.float32)
            raw_weather = np.zeros((100, CFG.SEQ_LEN, CFG.NUM_WEATHER_FEATURES))

        # C. Temporal Shift & Scaling Augmentation (2x Data)
        aug_demand = raw_demand.copy()
        scales = np.random.uniform(0.8, 1.2, (aug_demand.shape[0], 1, CFG.NUM_NODES))
        aug_demand = aug_demand * scales
        for i, shift in enumerate(np.random.choice([-1, 0, 1], size=aug_demand.shape[0])):
            aug_demand[i] = np.roll(aug_demand[i], shift, axis=0)
            
        demand_full = np.concatenate([raw_demand, aug_demand], axis=0)
        weather_full = np.concatenate([raw_weather, raw_weather], axis=0)
        self.conditions = np.concatenate([raw_conditions, raw_conditions], axis=0)
        
        # Normalize & Format
        demand_norm = (demand_full - demand_full.mean()) / (demand_full.std() + 1e-8)
        self.data = np.concatenate([demand_norm, weather_full], axis=-1).astype(np.float32)
        print(f"   ✓ SOTA Dataset Ready: {self.data.shape} | Conditions: {self.conditions.shape}")

    def __len__(self): return len(self.data)
        
    def __getitem__(self, idx):
        return torch.from_numpy(self.data[idx]).permute(1, 0), torch.from_numpy(self.conditions[idx])

## 5 · Model Architecture (TCN + Self-Attention)

In [ ]:
class CausalConv1d(nn.Conv1d):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, dilation=1, groups=1, bias=True):
        self._causal_padding = (kernel_size - 1) * dilation
        super().__init__(in_channels, out_channels, kernel_size, stride=stride, padding=self._causal_padding, dilation=dilation, groups=groups, bias=bias)
    def forward(self, x):
        out = super().forward(x)
        return out[:, :, :-self._causal_padding] if self._causal_padding != 0 else out

class TCNBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, dropout):
        super().__init__()
        self.net = nn.Sequential(
            CausalConv1d(n_inputs, n_outputs, kernel_size, stride=stride, dilation=dilation), nn.ReLU(), nn.Dropout(dropout),
            CausalConv1d(n_outputs, n_outputs, kernel_size, stride=stride, dilation=dilation), nn.ReLU(), nn.Dropout(dropout),
        )
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
    def forward(self, x):
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(self.net(x) + res)

class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs, num_channels, kernel_size, dropout):
        super().__init__()
        layers = []
        for i, out_ch in enumerate(num_channels):
            in_ch = num_inputs if i == 0 else num_channels[i - 1]
            layers.append(TCNBlock(in_ch, out_ch, kernel_size, stride=1, dilation=2**i, dropout=dropout))
        self.network = nn.Sequential(*layers)
    def forward(self, x): return self.network(x)

class GenerativeCounterfactualVAE(nn.Module):
    def __init__(self):
        super().__init__()
        tcn_flat = CFG.SEQ_LEN * CFG.TCN_CHANNELS[-1]
        self.encoder_tcn = TemporalConvNet(CFG.NUM_FEATURES, CFG.TCN_CHANNELS, CFG.KERNEL_SIZE, CFG.DROPOUT)
        
        # Self-Attention Layer
        self.attention = nn.MultiheadAttention(embed_dim=CFG.TCN_CHANNELS[-1], num_heads=8, batch_first=True)
        
        self.fc_mu = nn.Linear(tcn_flat, CFG.LATENT_DIM)
        self.fc_logvar = nn.Linear(tcn_flat, CFG.LATENT_DIM)
        
        self.decoder_fc = nn.Sequential(
            nn.Linear(CFG.LATENT_DIM + CFG.COND_DIM, CFG.DECODER_HIDDEN), nn.ReLU(),
            nn.Linear(CFG.DECODER_HIDDEN, tcn_flat), nn.ReLU()
        )
        self.decoder_tcn = TemporalConvNet(CFG.TCN_CHANNELS[-1], [64, CFG.NUM_FEATURES], CFG.KERNEL_SIZE, CFG.DROPOUT)

    def encode(self, x):
        h = self.encoder_tcn(x)
        h_permuted = h.permute(0, 2, 1) 
        attn_out, _ = self.attention(h_permuted, h_permuted, h_permuted)
        h_flat = attn_out.flatten(start_dim=1)
        return self.fc_mu(h_flat), self.fc_logvar(h_flat)

    def reparameterize(self, mu, logvar):
        return mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)

    def decode(self, z, condition):
        h = self.decoder_fc(torch.cat([z, condition], dim=-1)).view(-1, CFG.TCN_CHANNELS[-1], CFG.SEQ_LEN)
        return self.decoder_tcn(h)

    def forward(self, x, condition):
        mu, logvar = self.encode(x)
        return self.decode(self.reparameterize(mu, logvar), condition), mu, logvar

def vae_loss_function(recon_x, x, mu, logvar, current_kld_weight: float = 0.0, physics_loss=torch.tensor(0.0)):
    recon = F.mse_loss(recon_x, x, reduction="mean")
    kld = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + (current_kld_weight * kld) + physics_loss

## 6 · Checkpoint-Resilient Training Engine

In [ ]:
def train_model():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n🚀 Training initiated on: {device}")
    
    model = GenerativeCounterfactualVAE().to(device)
    optimizer = optim.Adam(model.parameters(), lr=CFG.LEARNING_RATE)
    loader = DataLoader(EVDemandDataset(), batch_size=CFG.BATCH_SIZE, shuffle=True)
    
    start_epoch = 1
    
    # --- RESUME FROM GOOGLE DRIVE CHECKPOINT ---
    if os.path.exists(CFG.CHECKPOINT_FILE):
        print(f"\n🔍 Found existing checkpoint in Google Drive!")
        checkpoint = torch.load(CFG.CHECKPOINT_FILE, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        print(f"🔄 Resuming training from Epoch {start_epoch}...")
    else:
        print("🆕 Starting fresh training run...")

    model.train()
    for epoch in range(start_epoch, CFG.EPOCHS + 1):
        epoch_loss = 0.0
        current_kld = min(0.1, (epoch - 1) / 50.0)

        for batch_x, batch_cond in loader:
            x, cond = batch_x.to(device), batch_cond.to(device)

            optimizer.zero_grad()
            recon, mu, logvar = model(x, cond)
            loss = vae_loss_function(recon, x, mu, logvar, current_kld_weight=current_kld, physics_loss=torch.tensor(0.0, device=device))
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP_NORM)
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / max(len(loader), 1)
        
        # Print stats every epoch for paper logs
        print(f"Epoch {epoch:>3}/{CFG.EPOCHS} | Avg Loss: {avg_loss:.6f} | KLD: {current_kld:.4f}")
        
        # --- PERIODIC GOOGLE DRIVE SAVING ---
        if epoch % 5 == 0 or epoch == CFG.EPOCHS:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': avg_loss,
            }, CFG.CHECKPOINT_FILE)
            print(f"   💾 Checkpoint saved to Drive at Epoch {epoch}")

    print("\n✅ Training complete! SOTA Checkpoint preserved in Google Drive.")
    return model

if __name__ == "__main__":
    trained_model = train_model()